# Metrics Walkthrough — Hiểu từng bước tính điểm

Notebook này đi từng phần nhỏ của `metrics.py`, chạy trên **dữ liệu giả** để bạn thấy chính xác điểm được tính thế nào — **không cần gọi LLM** cho phần rule-based.

Cấu trúc:
1. Setup & sample data
2. **Faithfulness Evaluator** — entity presence rule · DeepEval LLM · combined
3. **Expansion Evaluator** — length rule · GEval LLM · combined
4. **Marketing Vibe Evaluator** — FB structure · penalty · LLM 5 dims · combined
5. End-to-end: so sánh 3 samples
6. Thử nghiệm tự do

## 0. Setup

In [ ]:
import sys, os
from pathlib import Path
from dotenv import find_dotenv, load_dotenv

_dotenv_path = find_dotenv(usecwd=True)
load_dotenv(_dotenv_path, override=True, encoding="utf-8")
ROOT = Path(_dotenv_path).resolve().parent if _dotenv_path else Path.cwd().resolve()

if str(ROOT / "evaluation") not in sys.path:
    sys.path.insert(0, str(ROOT / "evaluation"))

import re, json
import numpy as np
import pandas as pd
from IPython.display import display

print("ROOT:", ROOT)

## 1. Sample data — 3 bài viết với chất lượng khác nhau

| | Mô tả |
|---|---|
| `GOOD` | Bài F&B tốt: có giá, deal, CTA rõ, nhịp tự nhiên |
| `GENERIC` | Bài generic: không có giá, hype, CTA mờ |
| `BAD` | Bài kém: AI opener, hype blocklist, không CTA |

In [ ]:
SEED = """
Moon Oven Bakery — deal cuối tuần:
sản phẩm: Signature Weekend Box
- Combo 6 bánh ngọt (tổng 420g), gói hộp "Weekend Edition"
- Giảm 25%, chỉ còn 149.000đ
- Freeship trong 3km
- Áp dụng thứ 7 và chủ nhật
- Đặt qua app MoMo hoặc nhắn inbox
"""

TITLE = "Viết bài Facebook cho deal cuối tuần của Moon Oven Bakery"

OUTPUT_GOOD = """
[CUỐI TUẦN NÀY ĂN BÁNH CHO ĐÁNG YÊU NHA ✨]

Team mê bánh ơi, Moon Oven Bakery vừa bật deal cuối tuần:
Signature Weekend Box — combo 6 bánh ngọt 420g, giảm 25% chỉ còn 149.000đ.

Gói hộp "Weekend Edition" xinh xắn, freeship trong 3km.

Deal chỉ có thứ 7 và chủ nhật thôi.
Đặt qua app MoMo hoặc nhắn inbox để chốt ngay.
"""

OUTPUT_GENERIC = """
Moon Oven Bakery có chương trình deal cuối tuần với sản phẩm Signature Weekend Box.
Combo gồm 6 bánh ngọt thủ công, tổng 420g, đóng trong gói hộp "Weekend Edition".

Giá giảm 25%, còn 149.000đ. Freeship áp dụng trong phạm vi 3km — số lượng có hạn mỗi ngày.

Chương trình chỉ áp dụng vào thứ 7 và chủ nhật.
Đặt hàng qua app MoMo hoặc nhắn inbox của Moon Oven Bakery.
"""
# "thủ công" và "số lượng có hạn mỗi ngày" không có trong seed → LLM sẽ flag fabrication → llm_faithfulness ~0.6–0.8

OUTPUT_BAD = """
Trong bối cảnh ngành F&B ngày càng phát triển, Moon Oven Bakery tự hào là thương hiệu
tiên phong trong lĩnh vực bánh ngọt cao cấp tại Việt Nam.

Chỉ với 149.000đ, đây chắc chắn là cơ hội vàng bạn không thể bỏ lỡ.
Với đỉnh cao công nghệ làm bánh và nguyên liệu nhập khẩu hoàn hảo nhất,
chúng tôi mang đến sự đột phá trong trải nghiệm ẩm thực của bạn.

Hãy cùng khám phá ngay hôm nay!
"""
# Chỉ có 1 entity: 149.000đ — bỏ hết entity còn lại + AI opener + nhiều hype words → faithfulness thấp + penalty cao

samples = {
    "GOOD":    OUTPUT_GOOD,
    "GENERIC": OUTPUT_GENERIC,
    "BAD":     OUTPUT_BAD,
}
print("Sample data loaded — 3 bài: GOOD / GENERIC / BAD")

---
## 2. Faithfulness Evaluator

Đánh giá mức độ **trung thành với nội dung mồi (seed)** — output có giữ nguyên các entity quan trọng và không bịa claim mới không?

### 2.1 Rule: Entity Presence Score

**Logic:** Trích entity từ seed (giá, thông số, tên trong ngoặc kép) → kiểm tra % entity xuất hiện trong output.

```
rule_entity_score = matched_entities / total_entities
```

In [ ]:
from metrics import EntityPresenceRule
import re

rule = EntityPresenceRule()

# Một ví dụ cho mỗi unit trong PRICE_RE
price_examples = [
    "giảm còn 149.000đ",        # đ
    "chỉ 2.500.000 VNĐ",        # VNĐ
    "giá 89vnđ mỗi ly",         # vnđ
    "giá 89vnd mỗi ly",         # vnd
    "80.000 đồng một ly",       # đồng
    "combo 50USD",               # USD
    "chỉ 5$",                   # $
    "giá 1.2 triệu",            # triệu
    "chỉ 1.5tr cho cả set",     # tr
    "deal 99k mỗi suất",        # k
    "tiết kiệm 25%",            # %
    "freeship trong 3km",       # ✗ không match — km không có trong list
]
print("=== PRICE_RE — giá tiền + discount ===")
print("Units: đ | VNĐ | vnđ | vnd | đồng | USD | $ | triệu | tr | k | %\n")
for s in price_examples:
    hits = rule.extract_prices(s)
    print(f"  {'✓' if hits else '✗'} {s!r:40s} → {hits}")

In [ ]:
# Một ví dụ cho mỗi unit trong SPEC_RE
spec_examples = [
    "ly 500ml",                  # ml
    "bình 1 lít trà",            # lít
    "chai 1.5 lit nước",         # lit
    "bình 2l nước",              # l
    "espresso 30cc",             # cc
    "thịt 1.2kg",                # kg
    "bánh 420g",                 # g
    "khẩu phần 650 kcal",        # kcal
    "đốt 200 cal mỗi ngày",      # cal
    "giảm 25%",                  # ✗ không match — % đã chuyển sang PRICE_RE
    "freeship trong 3km",        # ✗ không match — km không có trong list
]
print("=== SPEC_RE — thông số F&B + đơn vị đo ===")
print("Units: ml | lít | lit | l | cc | kg | g | kcal | cal  (% → PRICE_RE)\n")
for s in spec_examples:
    hits = rule.extract_specs(s)
    print(f"  {'✓' if hits else '✗'} {s!r:40s} → {hits}")

In [ ]:
# Một ví dụ cho mỗi pattern trong COMBO_RE
combo_examples = [
    "Combo 6 bánh ngọt",         # combo \d+
    "Set 2 người ăn no",          # set \d+
    "Deal 3 món hấp dẫn",         # deal \d+
    "Gói 5 buổi tư vấn",          # gói \d+
    "order 2 món kèm nước",       # \d+ món
    "mua 3 ly tặng 1",            # \d+ ly
    "gọi 4 tô bún bò",            # \d+ tô
    "đặt 2 bát phở",              # \d+ bát
    "thêm 1 đĩa rau",             # \d+ đĩa
    "combo 3 phần cơm",           # \d+ phần
    "set 2 suất ăn trưa",         # \d+ suất
    "bàn 6 người",                # \d+ người
    "mua 10 cái bánh",            # ✗ không match — cái không có trong list
]
print("=== COMBO_RE — combo/deal/serving patterns F&B ===")
print("Prefix: combo | set | deal | gói  +  số")
print("Suffix: số  +  món | ly | tô | bát | đĩa | phần | suất | người\n")
for s in combo_examples:
    hits = rule.extract_combos(s)
    print(f"  {'✓' if hits else '✗'} {s!r:40s} → {hits}")

In [ ]:
# Một ví dụ cho mỗi loại ngoặc kép trong QUOTED_RE (straight " và curly " ")
quoted_examples = [
    ('"Weekend Edition"',          "straight \""),
    ('“Weekend Edition”', "curly “”"),
    ('"Trà Sữa Ô Long 500ml"',    "straight \""),
    ('“Trà Sữa Ô Long 500ml”', "curly “”"),
    ('"x"',                        "✗ quá ngắn (< 2 ký tự)"),
    ('"' + 'a' * 81 + '"',         "✗ quá dài (> 80 ký tự)"),
]
print("=== QUOTED_RE — chuỗi trong ngoặc kép (2–80 ký tự) ===\n")
for s, label in quoted_examples:
    hits = rule.extract_quoted(s)
    print(f"  {'✓' if hits else '✗'} [{label:22s}] {s[:45]!r:47s} → {hits}")

In [ ]:
# Một ví dụ cho mỗi key trong NAME_RE
# Keys: tên | món | combo | set | menu | sản phẩm | thương hiệu | quán
name_examples = [
    "tên: Combo Thứ 7",
    "món: Trà Ô Long 500ml",
    "combo: Bánh Mì Thịt Nướng",
    "set: Breakfast For Two",
    "menu: Cơm Sườn Đặc Biệt",
    "sản phẩm: Signature Weekend Box",
    "thương hiệu: Moon Oven Bakery",
    "quán: The Coffee House. Chi nhánh Q1",   # cắt tại "." → chỉ lấy phần trước
    "giá: 149.000đ",                          # ✗ "giá" không có trong list
    "model: iPhone 16 Pro",                   # ✗ "model" đã bỏ
]
print("=== NAME_RE — tên sản phẩm / thương hiệu / món F&B ===")
print("Keys: tên | món | combo | set | menu | sản phẩm | thương hiệu | quán\n")
for s in name_examples:
    hits = rule.extract_names(s)
    print(f"  {'✓' if hits else '✗'} {s!r:50s} → {hits}")

In [ ]:
# Tổng hợp — entities thực sự được trích từ SEED (5 rule cộng lại)
print("=== Entities trích từ SEED (tổng hợp 5 rule) ===\n")
entities = rule.extract_entities(SEED)
for e in entities:
    print(f"  · {e!r}")
print(f"\nTổng: {len(entities)} entity — đây là những thứ EntityPresenceRule.measure sẽ tìm trong output")

In [ ]:
rows = []
for name, output in samples.items():
    score, detail = rule.measure(SEED, output)
    rows.append({
        "sample": name,
        "rule_entity_score": round(score, 3),
        "matched": detail["matched"],
        "total_entities": len(detail["entities"]),
    })

display(pd.DataFrame(rows))
print("\nGiải thích: GOOD giữ đủ 6/6 entity → score=1.0. GENERIC giữ 4/6 → score≈0.67. BAD chỉ giữ 1/6 → score≈0.17.")

### 2.2 LLM Score (DeepEval FaithfulnessMetric)

**Cần LLM judge.** Skip cell này nếu chưa setup Azure.

DeepEval kiểm tra: output có claim nào **mâu thuẫn hoặc bịa thêm** so với `retrieval_context` (seed) không.

In [5]:
# Setup judge — chỉ cần chạy 1 lần
from openai import AzureOpenAI
from metrics import JudgeLLM, FaithfulnessEvaluator, FaithfulnessGEval

def _env(k): return (os.getenv(k) or "").strip()

_client = AzureOpenAI(
    azure_endpoint=_env("JUDGE_AZURE_ENDPOINT") or _env("AZURE_OPENAI_ENDPOINT"),
    api_key=_env("JUDGE_AZURE_API_KEY") or _env("AZURE_OPENAI_API_KEY"),
    api_version=_env("JUDGE_AZURE_API_VERSION") or _env("OPENAI_API_VERSION") or "2024-08-01-preview",
)
_model_id = _env("JUDGE_MODEL") or _env("OPENAI_MODEL") or "gpt-4o-mini"
judge_llm = JudgeLLM(_client, _model_id)
print("Judge:", judge_llm.get_model_name())

Judge: gpt-5.4


In [ ]:
f_geval = FaithfulnessGEval(judge_llm)
actual_output = samples["GENERIC"]
score, explanation = f_geval.measure(TITLE, SEED, actual_output)

print("Title:", TITLE)
print("Seed:", SEED)
print(f"Output: {actual_output}\n")
print(f"Score: {score:.3f}\nExplanation: {explanation}")

In [ ]:
f_ev = FaithfulnessEvaluator(judge_llm)

rows = []
for name, output in samples.items():
    r = f_ev.evaluate_one(TITLE, SEED, output)
    rows.append({
        "sample":              name,
        "rule_entity_score":   round(r.rule_entity_score, 3),
        "llm_faithfulness":    round(r.llm_faithfulness_score, 3),
        "combined (50/50)":    round(r.combined_score, 3),
        "llm_reason":          r.llm_reason,
    })
with pd.option_context('display.max_colwidth', None):
    display(pd.DataFrame(rows))

---
## 3. Expansion Evaluator

Đánh giá mức độ **mở rộng có chất lượng** từ seed thành bài viết hoàn chỉnh — output có diễn giải lợi ích và thêm ngữ cảnh thực tế không, hay chỉ lặp lại bullet kỹ thuật?

### 3.1 Rule: Content Expansion Score

**Logic:** Đo 3 tín hiệu thực tế thay vì chỉ đo độ dài thô:

| Tín hiệu | Trọng số | Ý nghĩa |
|---|---|---|
| Benefit language (`giúp`, `tận hưởng`, `mang lại`…) | 35% | Từ tính năng → lợi ích |
| Usage context (`cuối tuần`, `cùng bạn bè`, `thư giãn`…) | 25% | Có ngữ cảnh sử dụng thực tế |
| Length ratio (target 1.5x–2.5x) | 40% | Output mở rộng hợp lý |

```
content_score = 0.4 × length_score + 0.35 × benefit_score + 0.25 × context_score
```

In [ ]:
from metrics import ContentExpansionRule, ExpansionGEval, ExpansionQualityEvaluator

rule_content = ContentExpansionRule()

rows = []
for name, output in samples.items():
    ratio = (len(output.strip()) + 1) / (len(SEED.strip()) + 1)
    score = rule_content.measure(SEED, output)
    rows.append({
        "sample":         name,
        "seed_len":       len(SEED.strip()),
        "output_len":     len(output.strip()),
        "ratio":          round(ratio, 2),
        "content_score":  score,
    })

display(pd.DataFrame(rows))
print("\nNote: content_score chiếm 10% của expansion_combined.")

### 3.2 LLM Score (GEval) + Combined

GEval hỏi judge: bài có **diễn giải lợi ích** cho khách hàng không, có **ngữ cảnh thực tế** không, hay chỉ lặp lại bullet kỹ thuật?

```python
expansion_combined = 0.9 × llm_expansion + 0.1 × content_rule
```

In [6]:
e_geval = ExpansionGEval(judge_llm)
actual_output = samples["GOOD"]
score, explanation = e_geval.measure(TITLE, SEED, actual_output)

print("Title:", TITLE)
print("Seed:", SEED)
print(f"Output: {actual_output}\n")
print(f"Score: {score:.3f}\nExplanation: {explanation}")

c:\Users\vqnhan\AppData\Local\Programs\Python\Python314\Lib\site-packages\rich\live.py:260: UserWarning: install 
"ipywidgets" for Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')

error uploading: ('Connection aborted.', ConnectionResetError(10054, 'An existing connection was forcibly closed by
the remote host', None, 10054, None))

Title: Viết bài Facebook cho deal cuối tuần của Moon Oven Bakery
Seed: 
Moon Oven Bakery — deal cuối tuần:
sản phẩm: Signature Weekend Box
- Combo 6 bánh ngọt (tổng 420g), gói hộp "Weekend Edition"
- Giảm 25%, chỉ còn 149.000đ
- Freeship trong 3km
- Áp dụng thứ 7 và chủ nhật
- Đặt qua app MoMo hoặc nhắn inbox

Output: 
[CUỐI TUẦN NÀY ĂN BÁNH CHO ĐÁNG YÊU NHA ✨]

Team mê bánh ơi, Moon Oven Bakery vừa bật deal cuối tuần:
Signature Weekend Box — combo 6 bánh ngọt 420g, giảm 25% chỉ còn 149.000đ.

Gói hộp "Weekend Edition" xinh xắn, freeship trong 3km.

Deal chỉ có thứ 7 và chủ nhật thôi.
Đặt qua app MoMo hoặc nhắn inbox để chốt ngay.


Score: 0.800
Explanation: Actual Output bám rất sát Input và bao quát đầy đủ thông tin trong Context: đúng deal cuối tuần của Moon Oven Bakery, nêu đủ Signature Weekend Box, combo 6 bánh 420g, hộp Weekend Edition, giảm 25% còn 149.000đ, freeship 3km, áp dụng thứ 7 và chủ nhật, và kênh đặt MoMo/inbox. Tuy nhiên bài chủ yếu dừng ở mức thông báo/liệt kê ưu đãi

In [ ]:
e_ev = ExpansionQualityEvaluator(judge_llm)

rows = []
for name, output in samples.items():
    r = e_ev.evaluate_one(TITLE, SEED, output)
    rows.append({
        "sample":            name,
        "llm_expansion":     round(r.llm_expansion_score, 3),
        "rule_content":      round(r.rule_length_score, 3),
        "combined (90/10)":  round(r.combined_score, 3),
        "reason":            r.llm_reason,
    })

with pd.option_context('display.max_colwidth', None):
    display(pd.DataFrame(rows))
print("\nCông thức: combined = 0.9 × llm + 0.1 × content_rule")

---
## 4. Marketing Vibe Evaluator

Đánh giá **chất lượng cảm giác tổng thể** của bài viết theo tiêu chuẩn F&B Facebook-native: giọng thật, có số liệu, thuyết phục bằng specifics, CTA rõ, nhịp tự nhiên.

### 4.1 Rule: FB Structure Score

Chỉ dùng regex, không cần LLM.

| Tiêu chí | Max điểm |
|---|---|
| ≥60% đoạn ≤320 ký tự | 0.35 |
| Avg đoạn ≤280 ký tự | 0.25 |
| Có bullet list | 0.20 |
| Có CTA line | 0.20 |

In [ ]:
from metrics import rule_fb_structure

rows = []
for name, output in samples.items():
    score, detail = rule_fb_structure(output)
    rows.append({
        "sample":          name,
        "fb_structure":    round(score, 3),
        "avg_para_len":    round(detail["avg_len"], 1),
        "short_ratio":     round(detail["short_ratio"], 2),
        "has_list":        detail["has_list"],
        "has_cta_line":    detail["has_cta_line"],
    })

display(pd.DataFrame(rows))

# Breakdown điểm thủ công cho GOOD
score, d = rule_fb_structure(OUTPUT_GOOD)
sr = d["short_ratio"]
al = d["avg_len"]
p_short = 0.35 if sr >= 0.6 else 0.35 * sr
p_len   = 0.25 if al <= 280 else max(0.0, 0.25 * (1 - (al - 280) / 400))
p_list  = 0.20 if d["has_list"] else 0.0
p_cta   = 0.20 if d["has_cta_line"] else 0.0
print(f"\nBreakdown GOOD:  short_ratio={p_short:.2f}  avg_len={p_len:.2f}  list={p_list:.2f}  cta={p_cta:.2f}  → total={p_short+p_len+p_list+p_cta:.3f}")

### 4.2 Rule: Penalty

Phát hiện pattern xấu và trừ điểm thẳng vào `vibe_combined`.

| Lỗi | Trừ |
|---|---|
| ≥2 hype words | −0.10 |
| AI opener | −0.08 |
| Quá nhiều emoji (>1/40 chars) | −0.05 |
| Bullet templated | −0.05 |

In [ ]:
from metrics import rule_vibe_penalty, HYPE_BLOCKLIST, AI_OPENERS

print("=== HYPE BLOCKLIST ===")
for p in HYPE_BLOCKLIST:
    print(f"  {p}")

print("\n=== AI OPENERS ===")
for p in AI_OPENERS:
    print(f"  {p}")

In [ ]:
rows = []
for name, output in samples.items():
    penalty, detail = rule_vibe_penalty(output)
    rows.append({
        "sample":           name,
        "total_penalty":    round(penalty, 2),
        "hype_hits":        detail["hype_hits"],
        "ai_opener":        detail["ai_open"],
        "over_emoji":       detail["over_emoji"],
        "bullet_templated": detail["bullet_templated"],
    })

display(pd.DataFrame(rows))
print("\nBAD có AI opener + nhiều hype words → bị trừ nhiều nhất.")

### 4.3 LLM: 5 Dimensions

Judge trả JSON với 5 chiều (0–4), normalize về 0–1, nhân trọng số:

| Chiều | Weight |
|---|---|
| D1 VOICE_AUTHENTICITY | **0.25** |
| D2 BUSINESS_GROUNDING | 0.15 |
| D3 PERSUASION_GROUND | **0.25** |
| D4 CTA_QUALITY | 0.20 |
| D5 FB_NATIVE_FLOW | 0.15 |

In [ ]:
# Mô phỏng LLM trả về JSON — để hiểu cách tính mà không cần gọi API
from metrics import VIBE_WEIGHTS

mock_scores = {
    "GOOD":    {"d1": 3, "d2": 3, "d3": 3, "d4": 4, "d5": 3},
    "GENERIC": {"d1": 1, "d2": 1, "d3": 2, "d4": 1, "d5": 2},
    "BAD":     {"d1": 1, "d2": 0, "d3": 1, "d4": 0, "d5": 1},
}

rows = []
for name, dims in mock_scores.items():
    norm = {k: v / 4 for k, v in dims.items()}
    llm_vibe = sum(VIBE_WEIGHTS[k] * norm[k] for k in VIBE_WEIGHTS)
    rows.append({
        "sample":  name,
        "D1 (×0.25)": f"{dims['d1']}/4 → {norm['d1']:.2f}",
        "D2 (×0.15)": f"{dims['d2']}/4 → {norm['d2']:.2f}",
        "D3 (×0.25)": f"{dims['d3']}/4 → {norm['d3']:.2f}",
        "D4 (×0.20)": f"{dims['d4']}/4 → {norm['d4']:.2f}",
        "D5 (×0.15)": f"{dims['d5']}/4 → {norm['d5']:.2f}",
        "llm_vibe":   round(llm_vibe, 3),
    })

display(pd.DataFrame(rows))
print("\nllm_vibe = Σ(weight × norm_score)")

### 4.4 Combined

```python
vibe_combined = clip(0, 1,  0.85 × llm_vibe  +  0.15 × fb_structure  −  penalty)
```

In [ ]:
rows = []
for name, output in samples.items():
    dims = mock_scores[name]
    norm = {k: v / 4 for k, v in dims.items()}
    llm_vibe = sum(VIBE_WEIGHTS[k] * norm[k] for k in VIBE_WEIGHTS)

    fb_s, _  = rule_fb_structure(output)
    pen, _   = rule_vibe_penalty(output)

    combined = max(0.0, min(1.0, 0.85 * llm_vibe + 0.15 * fb_s - pen))

    rows.append({
        "sample":       name,
        "llm_vibe":     round(llm_vibe, 3),
        "fb_structure": round(fb_s, 3),
        "penalty":      round(pen, 3),
        "0.85×llm":     round(0.85 * llm_vibe, 3),
        "0.15×fb":      round(0.15 * fb_s, 3),
        "−penalty":     round(-pen, 3),
        "vibe_combined": round(combined, 3),
    })

display(pd.DataFrame(rows))
print("\nCông thức: vibe_combined = clip(0,1, 0.85×llm_vibe + 0.15×fb_structure − penalty)")

### 4.5 Gọi MarketingVibeEvaluator thật (cần LLM)

Cell trước dùng `mock_scores`. Cell này gọi judge thật để thấy LLM trả dims ra sao.

In [ ]:
from metrics import MarketingVibeEvaluator

m_ev = MarketingVibeEvaluator(judge_llm)

rows = []
for name, output in samples.items():
    r = m_ev.evaluate_one(TITLE, SEED, output)
    rows.append({
        "sample":          name,
        "llm_vibe":        round(r.llm_hook_tone_score, 3),
        "fb_structure":    round(r.rule_structure_score, 3),
        "vibe_combined":   round(r.combined_score, 3),
        "reason":          r.llm_reason[:150],
    })

display(pd.DataFrame(rows))

---
## 5. End-to-end: run_batch trên 3 samples

Chạy toàn bộ pipeline như notebook eval thật, so sánh 3 bài cùng lúc.

In [ ]:
from metrics import run_batch

cases = [
    {"case_id": name, "input_title": TITLE, "seed_content": SEED, "actual_output": output}
    for name, output in samples.items()
]

df = run_batch(cases, judge_llm, progress_chunk=3)
df.insert(0, "sample", [c["case_id"] for c in cases])

summary_cols = ["sample", "faithfulness_combined", "expansion_combined", "vibe_combined"]
display(df[summary_cols].round(3))

print("\n=== Overall mean per sample ===")
for _, row in df[summary_cols].iterrows():
    overall = np.mean([row["faithfulness_combined"], row["expansion_combined"], row["vibe_combined"]])
    print(f"  {row['sample']:8s} → {overall:.3f}")

---
## 6. Thử nghiệm tự do — chỉnh output và xem điểm thay đổi

Thay `MY_OUTPUT` bằng bất kỳ bài viết nào để kiểm tra ngay.

In [ ]:
MY_OUTPUT = """
# Dán bài viết của bạn vào đây
"""

if MY_OUTPUT.strip() and "Dán bài" not in MY_OUTPUT:
    # Rule scores — không cần LLM
    entity_score, entity_detail = rule.measure(SEED, MY_OUTPUT)
    fb_score, fb_detail         = rule_fb_structure(MY_OUTPUT)
    penalty, pen_detail         = rule_vibe_penalty(MY_OUTPUT)

    print("=== Rule Scores (không cần LLM) ===")
    print(f"  entity_presence : {entity_score:.3f}  matched={entity_detail['matched']}")
    print(f"  fb_structure    : {fb_score:.3f}  {fb_detail}")
    print(f"  vibe_penalty    : {penalty:.3f}  {pen_detail}")
else:
    print("Hãy thay MY_OUTPUT bằng bài viết thực để xem điểm.")